# Advanced Architectures and Applications in Agentic AI
## CA1 Assessment: Adaptive Customer Service Harness

Student name: Maximiliano Herrera </br>
Student ID: d00296830

## Dataset Preparation

### Load and Analysis

In [75]:
import pandas as pd

df = pd.read_csv('customer-dataset.csv')
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [76]:
categories = set(df['category'])
print(categories)
print(len(categories))

{'DELIVERY', 'SUBSCRIPTION', 'SHIPPING', 'CANCEL', 'PAYMENT', 'INVOICE', 'FEEDBACK', 'CONTACT', 'ACCOUNT', 'ORDER', 'REFUND'}
11


In [77]:
df.groupby(["category"]).size().reset_index(name="count")

,category,count
0,ACCOUNT,5986
1,CANCEL,950
2,CONTACT,1999
3,DELIVERY,1994
4,FEEDBACK,1997
5,INVOICE,1999
6,ORDER,3988
7,PAYMENT,1998
8,REFUND,2992
9,SHIPPING,1970


In [78]:
df.groupby(["category", "intent"]).size().reset_index(name="count")

,category,intent,count
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995


### Filter dataset by categories and intents

In [79]:
categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
intents_selected = ['create_account', 'delete_account', 'edit_account',
                    'cancel_order', 'change_order', 'place_order',
                    'check_refund_policy', 'get_refund', 'track_refund'
                   ]
df_filtered = df[
    df['category'].isin(categories_selected) &
    df['intent'].isin(intents_selected)
].rename(columns={"Unnamed: 0": "id"})

print(f"df_filtered: {len(df_filtered)}, {type(df_filtered)}")

df_filtered: 8977, <class 'pandas.DataFrame'>


In [80]:
import plotly.express as px

def plot_number_records_per_intent(dataframe):
    fig = px.bar(
        dataframe['intent'].value_counts().reset_index(),
        x='intent',
        y='count',
        title='Number of Records per Intent',
        text_auto=True
    )

    fig.show()

plot_number_records_per_intent(df_filtered)

In [81]:
from pandas import DataFrame

def get_split_dataset(categories_selected, intents_selected, max_total_rows=60, random_state=42):
    intent_count = len(intents_selected)
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent = (max_total_rows // intent_count) // 2 * 2
    rows_per_split = rows_per_intent // 2
    if rows_per_split < 1:
        raise ValueError("max_total_rows is too small for one train and one test row per intent")

    print(f"intents: {intent_count}")
    print(f"rows_per_intent: {rows_per_intent}")
    print(f"rows_per_split: {rows_per_split}")

    ds_filtered = df[
        df["category"].isin(categories_selected) &
        df["intent"].isin(intents_selected)
    ].rename(columns={"Unnamed: 0": "id"})
    print(f"ds_filtered: {len(ds_filtered)}")

    sampled = ds_filtered.groupby("intent", group_keys=False).sample(
        n=rows_per_intent, random_state=random_state
    )
    train_df = sampled.groupby("intent", group_keys=False).head(rows_per_split)
    test_df = sampled.drop(train_df.index)

    print(f"sampled: {len(sampled)}")
    print(f"train_df: {len(train_df)}")
    print(f"test_df: {len(test_df)}")
    return train_df, test_df

def get_dataset_batches_unbalanced(dataframe: DataFrame, batch_size=30, random_state=42):
    shuffled_df = (
        dataframe
        .drop_duplicates(subset=["instruction"]) # remove duplicated instructions
        .sample(frac=1, random_state=random_state) # shuffle records and use random_state for reproducibility
        .reset_index(drop=True) # reset index and do not insert it as new column
    )

    for start in range(0, len(shuffled_df), batch_size): #get a sequence from start (0, inclusive) to stop (len(shuffled_df), exclusive) with a step (batch_size)
        batch = shuffled_df.iloc[start:start + batch_size] # select batch_size rows starting from start

        #if batch is incomplete then break the loop
        if len(batch) < batch_size: 
            break

        # return a copy of the batch so original data is unchanged
        yield batch.copy()


def get_dataset_batches_balanced(dataframe: DataFrame, batch_size=30, random_state=42):    
    remaining = dataframe.drop_duplicates(subset=["instruction"]).copy()    
    num_intents = remaining["intent"].nunique()  
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent =  batch_size// num_intents

    if rows_per_intent == 0:
        raise ValueError("batch_size must be at least the number of intents")

    num_batches = remaining.groupby("intent").size().min() # rows per intent
    for _ in range(num_batches):
        batch = remaining.groupby("intent").sample(
            n=rows_per_intent,
            random_state=random_state
        )

        remaining= remaining.drop(batch.index)

        yield batch

def plot_andpreview_dataset(batch: DataFrame):
    plot_number_records_per_intent(batch)
    print(f"Num rows: {len(batch)}, num intents: {batch["intent"].nunique()}")
    print(len(batch))        
         

In [82]:
batches = get_dataset_batches_balanced(df_filtered,30)
train1_df =next(batches)
train2_df =next(batches)
test1_df =next(batches)
test2_df =next(batches)
# train_df, test_df = get_split_dataset(categories_selected, intents_selected, 50)

In [83]:
plot_andpreview_dataset(train1_df)
train1_df

Num rows: 27, num intents: 9
27


,id,flags,instruction,category,intent,response
860,860,BILZ,can you help me camceling the purchase I made?,ORDER,cancel_order,I've been informed that you need assistance wi...
79,79,BIQZ,whag do i need to do to cancel order {{Order N...,ORDER,cancel_order,I understand your need for assistance in cance...
479,479,BC,"I bought some product, I want to cancel order ...",ORDER,cancel_order,I've come to understand that you've made a pur...
1521,1521,BILQZ,how od i edit purchase {{Order Number}},ORDER,change_order,I see that you're looking to edit your purchas...
1164,1164,BKL,modify order {{Order Number}},ORDER,change_order,Thank you for bringing up the request to modif...
1784,1784,BLQ,want help to add an article to purchase {{Orde...,ORDER,change_order,We understand that you would like assistance w...
5974,5974,BLQ,assistance seeing ur money back guarantee,REFUND,check_refund_policy,I'll make it happen! I understand your request...
6426,6426,BLQ,i want help to see in which cases can i reques...,REFUND,check_refund_policy,Of course! I'm here to guide you through the s...
6720,6720,BL,help checking in what situations can I ask to ...,REFUND,check_refund_policy,Assuredly! I completely understand your desire...
10675,10675,BKL,creating new freemium account for daughter,ACCOUNT,create_account,It's wonderful to see your enthusiasm in creat...


In [84]:
plot_andpreview_dataset(train2_df)
train2_df

Num rows: 27, num intents: 9
27


,id,flags,instruction,category,intent,response
863,863,BLQ,need assistance to cancel purchase {{Order Num...,ORDER,cancel_order,I acknowledge that you're seeking assistance w...
94,94,B,I want help canceling order {{Order Number}},ORDER,cancel_order,I appreciate your request for assistance in ca...
791,791,BCEILNQZ,i cannot pay for these product how do i cancel...,ORDER,cancel_order,I've realized you're unable to make payment fo...
1527,1527,BL,I want assistance changing a product of order ...,ORDER,change_order,Thank you for reaching out! I completely under...
1824,1824,BLZ,I have to edit order {{Order Number}},ORDER,change_order,We understand that you need to edit order numb...
1318,1318,BLMNQ,i cant delete some items from order {{Order Nu...,ORDER,change_order,I'm conscious of the reality that you're facin...
6660,6660,BIL,in which situations can I ask for refunds?,REFUND,check_refund_policy,Unquestionably! I understand that you would li...
6056,6056,BLQ,wanna check in what cases can i request a refund,REFUND,check_refund_policy,I'll take care of it! I completely understand ...
6126,6126,BLZ,I need help seeing in which cases can I ask fo...,REFUND,check_refund_policy,I'll make it happen! I understand that you wou...
10824,10824,BLM,I want to know more about opening {{Account Ca...,ACCOUNT,create_account,Absolutely! I'm delighted to provide you with ...


In [85]:
plot_number_records_per_intent(test1_df)
test1_df

,id,flags,instruction,category,intent,response
78,78,BCL,I have got to cancel purchase {{Order Number}}...,ORDER,cancel_order,I've grasped that you need assistance with can...
321,321,BCILNPQ,i dont want purchase {{Order Number}} could i ...,ORDER,cancel_order,I grasp your desire to cancel your purchase wi...
801,801,BILZ,can you help me cancelling purchase {{Order N...,ORDER,cancel_order,I'll take care of it! I'm here to assist you w...
1528,1528,BLQ,have a problem with changing purchase {{Order ...,ORDER,change_order,We appreciate you contacting us regarding the ...
1825,1825,BILZ,acn you help me changing something of purchase...,ORDER,change_order,We understand that you would like assistance w...
1333,1333,BILQ,how can i get information about updating order...,ORDER,change_order,We understand your desire to obtain informatio...
6499,6499,BILQWZ,how to checkur goddamn money back guarantee,REFUND,check_refund_policy,Thank you for your inquiry about how to check ...
6057,6057,BQ,want assistance to check your refund policy,REFUND,check_refund_policy,I'll make it happen! I completely understand y...
6128,6128,BLQ,i have got to see in which cases can i request...,REFUND,check_refund_policy,Definitely! I understand your need for clarity...
10904,10904,BIL,can you help me opening a pro account?,ACCOUNT,create_account,Positively! I'm thrilled to assist you in open...


In [86]:
plot_number_records_per_intent(test2_df)
test2_df

,id,flags,instruction,category,intent,response
756,756,BLQZ,needassistance cancelling purchase {{Order Num...,ORDER,cancel_order,I've understood that you need assistance with ...
963,963,B,problem with canceling order {{Order Number}},ORDER,cancel_order,I realized that you're encountering difficulti...
221,221,BIL,what do I need to do to cancel purchase {{Orde...,ORDER,cancel_order,I gather that you're seeking guidance on cance...
1529,1529,BL,help editing order {{Order Number}},ORDER,change_order,Grateful you reached out to us for assistance ...
1655,1655,BIMQZ,ohw can i add some items to order {{Order Numb...,ORDER,change_order,Happy to get your message about adding items t...
1623,1623,BQ,need to change order {{Order Number}},ORDER,change_order,We appreciate your request to modify order num...
6348,6348,BL,I need help seeing how long reimbursements take,REFUND,check_refund_policy,Sure! I understand that you're eager to know h...
6058,6058,BL,I want help to check in which cases can I ask ...,REFUND,check_refund_policy,"I'm on the same wavelength, your need for clar..."
6130,6130,BILQZ,can i cxheck in which cases can i ask for refunds,REFUND,check_refund_policy,Assuredly! I completely understand your desire...
10499,10499,BKL,create new freemium account,ACCOUNT,create_account,We appreciate your interest in creating a new ...


## Test Harnes Agent

In [87]:
from general import clean_agentic_memory

clean_agentic_memory()

Cleaning agentic memory...
 * rmtree Skills...
path not exist ./skills, skipping...
 * .skills folder created...
 * Delete AGENT.md...
 * /AGENT.md file created...


### Test inference mode, wihtout agentic memory

In [88]:
import logging
from general import initialize_logging
from harness_agent.harness import AgentHarness
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
model = get_llm("openai/gpt-oss-120b")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=False, agentic_memory_disabled=True)
# # response = harness.invoke("see the available payment options")
# # response = harness.invoke("were can i cancel purchase {{Order Number}}")
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-01 17:58:36 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
2026-11-01 17:58:37 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:58:37 - harness.py - invoke - INFO - Harness Response: Sure, I can walk you through the process of canceling a purchase. To give you the most accurate instructions, could you let me know a few details?

1. **Where did you make the purchase?**  
   - Example: an online retailer (Amazon, eBay, etc.), a subscription service (Netflix, Spotify), a mobile app store (Apple App Store, Google Play), a physical store, etc.

2. **What type of product or service did you buy?**  
   - Physical goods, digital download, subscription, ticket, etc.

3. **Do you have an order/reference number or receipt handy?**  
   - This often speeds up the cancellation process.

4. **How long ago was the purchase?**  
 

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1ac2bfd2-0c7a-439c-a837-1d7236bd1659'),
  AIMessage(content='Sure, I can walk you through the process of canceling a purchase. To give you the most accurate instructions, could you let me know a few details?\n\n1. **Where did you make the purchase?**  \n   - Example: an online retailer (Amazon, eBay, etc.), a subscription service (Netflix, Spotify), a mobile app store (Apple App Store, Google Play), a physical store, etc.\n\n2. **What type of product or service did you buy?**  \n   - Physical goods, digital download, subscription, ticket, etc.\n\n3. **Do you have an order/reference number or receipt handy?**  \n   - This often speeds up the cancellation process.\n\n4. **How long ago was the purchase?**  \n   - Some merchants have a limited window (e.g., 24‑48\u202fhours) for cancellations.\n\nOnce I have that information, I can give you step‑by‑step instr

### Test multiturn

In [89]:
response = harness.invoke("Unfortunatelly I cannot give you none of those details...")
response

2026-11-01 17:58:41 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:58:41 - harness.py - invoke - INFO - Harness Response: I understand — you don’t have the order details handy, but you can still try to cancel or reverse the purchase. Below is a **step‑by‑step checklist** you can follow for most merchants, whether it’s an online store, a subscription service, or a mobile‑app purchase.

---

## 1. Locate Anything You Do Have
Even if you don’t have a full order number, try to gather **any** of the following:

| What to Look For | Where to Find It |
|------------------|------------------|
| Confirmation email (subject line often contains “Your Order #…”, “Receipt”, “Purchase Confirmation”) | Inbox, spam folder, or the email address you used for the purchase |
| Transaction on your bank/credit‑card statement (date, merchant name, amount) | Online banking app, paper statement |
| Order history pa

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1ac2bfd2-0c7a-439c-a837-1d7236bd1659'),
  AIMessage(content='Sure, I can walk you through the process of canceling a purchase. To give you the most accurate instructions, could you let me know a few details?\n\n1. **Where did you make the purchase?**  \n   - Example: an online retailer (Amazon, eBay, etc.), a subscription service (Netflix, Spotify), a mobile app store (Apple App Store, Google Play), a physical store, etc.\n\n2. **What type of product or service did you buy?**  \n   - Physical goods, digital download, subscription, ticket, etc.\n\n3. **Do you have an order/reference number or receipt handy?**  \n   - This often speeds up the cancellation process.\n\n4. **How long ago was the purchase?**  \n   - Some merchants have a limited window (e.g., 24‑48\u202fhours) for cancellations.\n\nOnce I have that information, I can give you step‑by‑step instr

In [90]:
response = harness.invoke("Can you remind me my first question please?")
response

2026-11-01 17:58:41 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:58:41 - harness.py - invoke - INFO - Harness Response: Your first question was:

**“Can you help me canceling the purchase I made?”** (you asked for assistance with canceling a purchase).


{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1ac2bfd2-0c7a-439c-a837-1d7236bd1659'),
  AIMessage(content='Sure, I can walk you through the process of canceling a purchase. To give you the most accurate instructions, could you let me know a few details?\n\n1. **Where did you make the purchase?**  \n   - Example: an online retailer (Amazon, eBay, etc.), a subscription service (Netflix, Spotify), a mobile app store (Apple App Store, Google Play), a physical store, etc.\n\n2. **What type of product or service did you buy?**  \n   - Physical goods, digital download, subscription, ticket, etc.\n\n3. **Do you have an order/reference number or receipt handy?**  \n   - This often speeds up the cancellation process.\n\n4. **How long ago was the purchase?**  \n   - Some merchants have a limited window (e.g., 24‑48\u202fhours) for cancellations.\n\nOnce I have that information, I can give you step‑by‑step instr

### Test training mode and enable agentic memory

In [91]:
from harness_agent.harness import AgentHarness
import logging
from llm import get_llm

initialize_logging(level=logging.INFO)

model = get_llm("qwen/qwen3.8-27b")
# model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-luna")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=True, agentic_memory_disabled=False)
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-01 17:58:41 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
2026-11-01 17:58:41 - manager.py - load_skills - WARNING - There is not skills folder on: ./skills
2026-11-01 17:58:42 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:58:42 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-01 17:58:42 - manager.py - load_skills - WARNING - There is not skills folder on: ./skills
2026-11-01 17:58:42 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:58:43 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"


BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'set_skill' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n<function=set_skill>\n<parameter=name>\npurchase-cancellation-assistance\n</parameter>\n<parameter=description>\nUse when a customer requests help canceling a purchase or order.\n</parameter>\n<parameter=content>\n# Purchase Cancellation Assistance\n\n## When to Use\n- Customer asks to cancel a purchase, order, or transaction.\n- Customer expresses intent to reverse a recent buy.\n\n## Procedure\n1. **Acknowledge the Request**\n   - Confirm you understand they need help canceling their purchase.\n   - Example phrasing: "I\'ve been informed that you need assistance with canceling the purchase you made. I\'m here to provide guidance and support throughout the process."\n\n2. **Gather Necessary Details**\n   - Ask for the order number or trans

### Test that multiturn should be disable on training mode

In [ ]:
response = harness.invoke("What was my first question?")
response

2026-11-01 17:46:11 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:46:12 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:46:12 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-01 17:46:12 - guidelines_optimizer_agent.py - invoke - INFO - GuidelinesOptimizerAgent executed successfully, number of tool calls: 1
2026-11-01 17:46:12 - harness.py - invoke - INFO - Harness Response: Your first question was: "What was my first question?"


{'messages': [HumanMessage(content='What was my first question?', additional_kwargs={}, response_metadata={}, id='7410df2a-cacd-406b-84b0-a0074e8ede47'),
  AIMessage(content='Your first question was: "What was my first question?"', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 1167, 'total_tokens': 1180, 'completion_time': 0.025860309, 'completion_tokens_details': None, 'prompt_time': 0.10111786, 'prompt_tokens_details': None, 'queue_time': 0.022038643, 'total_time': 0.126978169}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a19838-0690-7490-890a-5574aeddf456-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1167, 'output_tokens': 13, 'total_tokens': 1180})],
 'content': 'Your first question was: "What was my first question?"',
 'text': 'Your first question was: "What was

## Testing and Evaluation

In [ ]:
clean_agentic_memory()

Cleaning agentic memory...
 * rmtree Skills...
 * .skills folder created...
 * Delete AGENT.md...
 * /AGENT.md file created...


In [112]:
from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError

delete_if_exists = True

client = Client()

def create_dataset(name, description, delete_if_exists= False):
    try:
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                return dataset

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        return dataset       
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = 'CA1 - Customer Dataset'
dataset = create_dataset(
    name = dataset_name,
    description='Customer questions and responses',
    delete_if_exists= delete_if_exists
)

Dataset 'CA1 - Customer Dataset' already exists. Deleting existing dataset...
Dataset 'CA1 - Customer Dataset' created successfully.


In [ ]:
from pandas import DataFrame

def create_examples(dataset: Dataset, df: DataFrame, split_name: str = 'train'):
    print(f"Creating {len(df)} examples for split: {split_name}")

    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "category": row["category"],
                "intent": row["intent"],
                "instruction": row["instruction"],
            },
            outputs= {
                "response": row["response"]                         
            }
        )

        example_ids.append(example.id)

    print(f"Set split name: {split_name}")
    client.update_examples(example_ids=example_ids,splits=[split_name] * len(example_ids))

In [92]:
SPLIT_TRAIN_1 = 'train_1'
SPLIT_TRAIN_2 = 'train_2'
SPLIT_TEST_1 = 'test_1'
SPLIT_TEST_2 = 'test_2'
create_examples(dataset, train1_df, SPLIT_TRAIN_1)
create_examples(dataset, train2_df, SPLIT_TRAIN_2)
create_examples(dataset, test1_df, SPLIT_TEST_1)
create_examples(dataset, test2_df, SPLIT_TEST_2)

Creating 27 examples for split: train_1
Set split name: train_1
Creating 27 examples for split: train_2
Set split name: train_2
Creating 27 examples for split: test_1
Set split name: test_1
Creating 27 examples for split: test_2
Set split name: test_2


In [93]:
examples = client.list_examples(dataset_name=dataset_name)
import pandas as pd

def get_examples_by_split(dataset_name, split_name):    
    examples = list(
        client.list_examples(
            dataset_name=dataset_name,
            splits=[split_name]
        )
    )

    df_examples = pd.DataFrame([
        {
            **example.inputs, #unpack operator
            **example.outputs,
            "langsmith_id": str(example.id),
            "splits": example.metadata.get("dataset_split")
                    if example.metadata else None
        }
        for example in examples
    ])

    return {
        'name' : split_name,
        'examples' : examples,
        'df': df_examples
    }

# def get_examples_from_dataset(dataset_name):
#     train_split = get_examples_by_split(dataset_name, SPLIT_TRAIN_1N)
#     test_split = get_examples_by_split(dataset_name, SPLIT_TEST)
#     return {
#         'train_split': train_split,
#         'test_split': test_split,
#     }

# splits = get_examples_from_dataset(dataset_name)
# splits

split_train_1 = get_examples_by_split(dataset_name,SPLIT_TRAIN_1)
split_train_2 = get_examples_by_split(dataset_name,SPLIT_TRAIN_2)
split_test_1 = get_examples_by_split(dataset_name,SPLIT_TEST_1)
split_test_2 = get_examples_by_split(dataset_name,SPLIT_TEST_2)

In [ ]:
from harness_agent.harness import AgentHarness
from pydantic import BaseModel, Field
from llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

class ResponseEvaluatorSchema(BaseModel):
    """Class to define the schema for quality evaluation."""
    score: int = Field(description="An integer score from 1 to 10 indicating the quality of the generated answer.")
    comment: str = Field(description="A brief explanation of the score")

def evaluator_agent_response_quality(inputs: dict, outputs: dict, reference_outputs: dict, model_name:str) -> dict:
    """Evaluate the quality of the agent's generated response using LLM with a structured output."""  
    query = inputs["instruction"]
    agent_response = outputs['agent_response']
    reference_response = reference_outputs['response']

    system_eval_prompt = "You are a Expert Customer Service Agent Evaluator."
    eval_prompt = f"""Given a customer query, compare and evaluate the agent's generated answer against the expected response.

    Query:
    {query}

    Expected:
    {reference_response}

    Generated:
    {agent_response}

    Rules:
        * Score from 1 to 10, where 1 is bad and 10 is excelent quality of generated answer.
        * Provide a comment on the score.
        * Return structured output.
    """

    model = get_llm(model_name)
    result = model.with_structured_output(ResponseEvaluatorSchema).invoke([SystemMessage(system_eval_prompt), HumanMessage(eval_prompt)])
    
    evaluator_result = {"score": result.score, "comment": result.comment}
    logging.debug(f"evaluator_agent_response_quality: {evaluator_result}")

    return evaluator_result

def run_agent(query:str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool):      
    model = get_llm(model_name)
    harness = AgentHarness(model = model, training_dataset=training_dataset, training_mode=training_mode, agentic_memory_disabled=agentic_memory_disabled)
    # response = harness.invoke("see the available payment options")
    response = harness.invoke(query)
    agent_response = response['text']
    
    logging.debug("---------------------run_agent------------------------")
    logging.debug(f"response: {agent_response}")
    
    return agent_response

def target_function(inputs: str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool) -> dict:
    agent_response = run_agent(inputs["instruction"], model_name, training_dataset, training_mode, agentic_memory_disabled)
    logging.debug("---------------------target_function------------------------")
    logging.debug(f"agent_response: {agent_response}")
    
    return {"agent_response": agent_response}


In [ ]:
def evaluate_agent(model_name, training, disable_agentic_memory, split):
    dataframe = split['df']
    examples = split['examples']
    training_mode = "training" if training else "testing"
    agentic_memory_mode  = "memory_disabled" if disable_agentic_memory else "memory_on"

    logging.debug(f"Evaluate Agent --> mode: {training_mode}, agentic_memory_mode: {agentic_memory_mode}")

    client.evaluate(
        lambda inputs: target_function(inputs, model_name, dataframe, training, disable_agentic_memory),
        data= examples,
        evaluators=[lambda inputs, outputs, reference_outputs : evaluator_agent_response_quality( inputs, outputs, reference_outputs, model_name )],
        experiment_prefix=f"exp-{training_mode}-{agentic_memory_mode}-{model_name}",
        metadata={
            "model_name": model_name,
            "training_mode": training_mode,
            "agentic_memory": agentic_memory_mode,
            "split": split["name"]
        },
        #Training must run sequentially because there are read/write operations into files
        max_concurrency= 1 if training else 5 
    )
    
# I will not traion the agent in this way (off-line from LangSmith) because the experiments will give me more obvservability on agent's behavior
def train_agent(model_name, split):
    dataframe = split['df']
    
    for row in dataframe.itertuples():
        agent_response = run_agent(query= row["instruction"], model_name = model_name, training_dataset= dataframe, training_mode =True  , agentic_memory_disabled=False)
    

In [ ]:
#1 run in training mode with train split
#2 evaluate in test mode with test split (without agentic memory)
#3 evaluate in test mode with test split (with agentic memory)

# model_name = "openai/gpt-oss-120b"
# model_name = "deepseek"
# model_name = "gpt-6-luna"
# #1 evaluate in test mode with test split (without agentic memory)
# evaluate_agent(model_name, training=False, disable_agentic_memory=True, split=splits['test_split'])

from general import clean_agentic_memory

def evaluation_pipeline(models):
    for model in models:
        logging.info(f"Executing evaluation pipeline for model: {model}...")   
        clean_agentic_memory()   

        #Baseline: Evaluate agent on inference mode with agentic memory disabled
        evaluate_agent(model, training=False, disable_agentic_memory=True, split=split_test_1)
        
        #Training 1: Evaluate agent on training mode with agentic memory enabled        
        evaluate_agent(model, training=True, disable_agentic_memory=False, split=split_train_1)
        
        #Test 1 (after training 1): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        evaluate_agent(model, training=False, disable_agentic_memory=False, split=split_test_1)

        #Training 2: Evaluate agent on training mode with agentic memory enabled        
        evaluate_agent(model, training=True, disable_agentic_memory=False, split=split_train_2)
        
        #Test 2 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        evaluate_agent(model, training=False, disable_agentic_memory=False, split=split_test_1)
        
        #Test 3 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_2
        evaluate_agent(model, training=False, disable_agentic_memory=False, split=split_test_2)

In [ ]:
evaluation_pipeline([   
    # "qwen/qwen3.8-27b",
    "openai/gpt-oss-120b",
    "deepseek",
    "gpt-6-luna"]
)

Executing evaluation pipeline for model: openai/gpt-oss-120b...
Cleaning agentic memory...
 * rmtree Skills...
 * Delete AGENT.md...
Evaluate Agent --> mode: testing, agentic_memory_mode: memory_disabled
View the evaluation results for experiment: 'exp-testing-memory_disabled-openai/gpt-oss-120b-2fca042f' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/ea4fade0-564a-4a1d-a89d-3e7675917d8b/compare?selectedSessions=5ee2d25a-4bed-4bfc-9194-cf910c453f95


Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
middlewares: 0
------harness: Sure, I can help you with that. Could you please provide the actual order number (the series of letters and/or digits associated with your purchase) so I can give you the most accurate instructions? Once I have the order number, I’ll walk you through the cancellation process.
---------------------run_agent------------------------
response: Sure, I can help you with that. Could you please provide 

In [ ]:
#2 run in training mode with train split
evaluate_agent(model_name, training_mode=True, disable_agentic_memory=False, split=splits['train_split'])

View the evaluation results for experiment: 'exp-ce54f689' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1b7e0dbd-abb8-4fa8-ad25-1ebd5026077a/compare?selectedSessions=5f95581f-64da-4349-af3e-084fae9bf872


Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
middlewares: 2
before_agent 1 messages
before_agent 1 messages
request_supervisor: see the available payment options, refeerence: Example response: Thank you for your interest in exploring our available payment options! We take pride in offering a variety of payment methods to cater to our customers' needs. To view the available payment options, you can visit our website and navigate to the payment page. There, you will find a comprehensive list of the payment methods we accept, including credit/debit cards, PayPal, bank transfers, Apple Pay, and Google Wallet. If you have any questions or need further assistance with any of these options, please don't hesitate to reac

In [ ]:
#3 evaluate in test mode with test split (with agentic memory)
evaluate_agent(model_name, training_mode=False, disable_agentic_memory=False, split=splits['test_split'])

View the evaluation results for experiment: 'exp-f5c5ba6a' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/1b7e0dbd-abb8-4fa8-ad25-1ebd5026077a/compare?selectedSessions=b34f055f-809d-4e92-8736-ee1c858ef4e0


Initializing AgentHarness - training_mode: False, agentic_memory_disabled: False
middlewares: 2
before_agent 1 messages
before_agent 1 messages
------harness: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, including credit/debit cards, PayPal, bank transfers, Apple Pay, and Google Wallet.

You can view the full list of accepted payment methods on our website by going to the Payment page.

If you have any questions or need help with a particular option, just let us know – we’re here to make the process as smooth as possible!
---------------------run_agent------------------------
response: Thank you for your interest in our payment options! We offer a variety of methods to suit your needs, inclu